<a href="https://colab.research.google.com/github/shivansh2310/Quantitative-Momentum/blob/main/Momentum_Crashes_and_the_Trend_Filter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### A. The "Bear Market Rally" Trap

Momentum is a fantastic strategy in a bull market. However, during a severe bear market (like 2008 or 2020), the "Winners" portfolio consists of the least-bad stocks, and the "Losers" portfolio consists of stocks that have been completely decimated.
When the market suddenly rebounds, the decimated "Losers" often rocket up 50% to 100% in a matter of weeks, while your "Winners" barely move. If you are running a standard Momentum portfolio during a market bottom, you will underperform massively. This is the classic "Momentum Crash" (Daniel and Moskowitz, 2016).

### B. Absolute vs. Relative Momentum

* Relative Momentum: "Buy AAPL because it is doing better than MSFT." (What we did yesterday).

* Absolute (Time-Series) Momentum: "Buy AAPL only if its current price is higher than its historical price."

### C. The 200-Day Trend Filter

To prevent catastrophic drawdowns, Gray and Vogel apply a strict time-series trend filter on top of the cross-sectional ranking. We look at the broader market (the S&P 500). If the S&P 500 is trading below its 200-day Simple Moving Average (SMA), it indicates a Bear Market regime. In this regime, human panic overrides standard behavioral anchoring, and we must move to cash or hedge the portfolio.

## Dual-Momentum Regime Filter

In [1]:
import pandas as pd
import numpy as np
import yfinance as yf
import warnings

warnings.filterwarnings('ignore')

In [2]:
universe = [
    'AAPL', 'MSFT', 'AMZN', 'NVDA', 'META', 'TSLA', 'GOOGL',
    'JPM', 'BAC', 'GS', 'MS', 'WFC', 'C',
    'XOM', 'CVX', 'COP', 'EOG', 'SLB',
    'JNJ', 'UNH', 'LLY', 'PFE', 'ABBV',
    'PG', 'KO', 'PEP', 'WMT', 'TGT',
    'CAT', 'DE', 'GE', 'HON', 'MMM'
]
market_proxy = 'SPY'

# Combine for downloading
download_list = universe + [market_proxy]

In [3]:
data = yf.download(download_list, period="2y")['Close']
data = data.dropna(axis=1)


[*********************100%***********************]  34 of 34 completed


In [4]:
spy_close = data[market_proxy]

In [5]:
# Calculate the 200-day Simple Moving Average (SMA)
spy_sma_200 = spy_close.rolling(window=200).mean()

In [6]:
# Determine the current regime (Latest Close vs Latest SMA)
latest_date = spy_close.index[-1]
current_spy_price = spy_close.loc[latest_date]
current_spy_sma = spy_sma_200.loc[latest_date]

is_bull_regime = current_spy_price > current_spy_sma

print("\n" + "="*60)
print(f" MACRO TREND REGIME (As of {latest_date.date()})")
print("="*60)
print(f"SPY Current Price:    ${current_spy_price:.2f}")
print(f"SPY 200-Day SMA:      ${current_spy_sma:.2f}")

if is_bull_regime:
    print("REGIME: BULL MARKET (Risk-On) 🟢")
    print("ACTION: Proceed with Cross-Sectional Momentum Allocation.")
else:
    print("REGIME: BEAR MARKET (Risk-Off) 🔴")
    print("ACTION: Move to CASH. High risk of Momentum Crash.")
print("="*60)


 MACRO TREND REGIME (As of 2026-06-22)
SPY Current Price:    $744.39
SPY 200-Day SMA:      $685.03
REGIME: BULL MARKET (Risk-On) 🟢
ACTION: Proceed with Cross-Sectional Momentum Allocation.


In [8]:
if is_bull_regime:
    print("\nCalculating Cross-Sectional Momentum (12m-1m)...")
    stock_data = data[universe] # Exclude SPY for the stock ranking

    # 252 days = 12 months, 21 days = 1 month
    price_t_minus_1m = stock_data.shift(21)
    price_t_minus_12m = stock_data.shift(252)

    academic_12m_1m_return = (price_t_minus_1m / price_t_minus_12m) - 1
    current_momentum = academic_12m_1m_return.loc[latest_date].dropna()

    # Rank and Sort
    momentum_df = pd.DataFrame({'Academic_12m_1m': current_momentum})
    momentum_df = momentum_df.sort_values('Academic_12m_1m', ascending=False)

    print("\n" + "="*50)
    print(" TARGET ALLOCATION: TOP 5 MOMENTUM WINNERS")
    print("="*50)
    display_df = momentum_df.head(5).copy()
    display_df['Academic_12m_1m'] = (display_df['Academic_12m_1m'] * 100).round(2).astype(str) + '%'
    print(display_df)
    print("="*50)
else:
    print("\n*** ALLOCATION HALTED ***")
    print("The broader market trend is negative. Holding the 'winners' in this environment")
    print("exposes the portfolio to severe downside risk and violent mean-reversion crashes.")
    print("Target Allocation: 100% Cash / T-Bills.")


Calculating Cross-Sectional Momentum (12m-1m)...

 TARGET ALLOCATION: TOP 5 MOMENTUM WINNERS
       Academic_12m_1m
Ticker                
CAT            145.14%
GOOGL          124.89%
SLB             63.29%
C               62.92%
GS              57.03%
